# **META MMS**

In [ ]:
!pip install scipy transformers torch

In [ ]:
import json
import os
import torch
import scipy.io.wavfile
from transformers import VitsModel, AutoTokenizer

print("Loading evaluation dataset...")
with open("hindi_evaluation_set.json", "r", encoding="utf-8") as f:
    eval_data = json.load(f)

print("Initializing Meta MMS Hindi Model (this will download the weights)...")
model_id = "facebook/mms-tts-hin"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = VitsModel.from_pretrained(model_id)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print(f"Using device: {device}")

# Create a new output directory so we don't mix them with XTTS results
output_dir = "mms_hindi_eval"
os.makedirs(output_dir, exist_ok=True)

print("\nStarting Audio Generation...")

# Generation Loop
for key, item in eval_data.items():
    text = item["text"]
    filename = f"{output_dir}/{item['id']}_mms.wav"

    print(f"Synthesizing {item['id']}...")

    # Tokenize the text
    inputs = tokenizer(text, return_tensors="pt").to(device)

    # Generate the waveform (VITS is highly efficient, this is usually very fast)
    with torch.no_grad():
        output = model(**inputs).waveform

    # Convert tensor to numpy array and save as .wav
    audio_data = output.squeeze().cpu().numpy()
    sample_rate = model.config.sampling_rate

    scipy.io.wavfile.write(filename, rate=sample_rate, data=audio_data)

print(f"\n✅ All done! Evaluation audio files have been saved to the '{output_dir}' folder.")

Loading evaluation dataset...
Initializing Meta MMS Hindi Model (this will download the weights)...


config.json:   0%|          | 0.00/1.64k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/907 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/145M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/762 [00:00<?, ?it/s]

Using device: cuda

Starting Audio Generation...
Synthesizing HIN_01...
Synthesizing HIN_02...
Synthesizing HIN_03...
Synthesizing HIN_04...
Synthesizing HIN_05...
Synthesizing HIN_06...
Synthesizing HIN_07...
Synthesizing HIN_08...
Synthesizing HIN_09...
Synthesizing HIN_10...
Synthesizing HIN_11...
Synthesizing HIN_12...
Synthesizing HIN_13...
Synthesizing HIN_14...
Synthesizing HIN_15...
Synthesizing HIN_16...
Synthesizing HIN_17...
Synthesizing HIN_18...
Synthesizing HIN_19...
Synthesizing HIN_20...

✅ All done! Evaluation audio files have been saved to the 'mms_hindi_eval' folder.


In [ ]:
import whisper
import jiwer
import json
import os
import pandas as pd
import warnings

# Suppress minor fp16 warnings
warnings.filterwarnings("ignore")

print("Loading evaluation dataset...")
with open("hindi_evaluation_set.json", "r", encoding="utf-8") as f:
    eval_data = json.load(f)

print("Loading Whisper 'medium' model...")
model = whisper.load_model("medium")

output_dir = "mms_hindi_eval"
results = []

print("\nStarting Transcription and Evaluation...\n")

# Evaluation Loop
for key, item in eval_data.items():
    file_id = item["id"]
    ground_truth = item["text"]
    audio_path = f"{output_dir}/{file_id}_mms.wav"

    if not os.path.exists(audio_path):
        print(f"Skipping {audio_path}: File not found.")
        continue

    # Transcribe
    transcription_result = model.transcribe(audio_path, language="hi")
    hypothesis = transcription_result["text"].strip()

    # Calculate Error Rates
    try:
        wer = jiwer.wer(ground_truth, hypothesis)
        cer = jiwer.cer(ground_truth, hypothesis)
    except ValueError:
        wer = 1.0
        cer = 1.0

    # Store data
    results.append({
        "Test_ID": file_id,
        "Category": key,
        "WER": round(wer, 3),
        "CER": round(cer, 3),
        "Original_Text": ground_truth,
        "Whisper_Transcription": hypothesis
    })

    print(f"[{file_id}] WER: {wer:.3f} | CER: {cer:.3f}")

# Create DataFrame
df_results = pd.DataFrame(results)

print("\n" + "="*40)
print("🏆 MMS EVALUATION SUMMARY")
print("="*40)

# Print average scores
print(f"Average WER: {df_results['WER'].mean():.3f}")
print(f"Average CER: {df_results['CER'].mean():.3f}")

# Save report
csv_filename = "mms_whisper_evaluation_report.csv"
df_results.to_csv(csv_filename, index=False)
print(f"\n✅ Detailed breakdown saved to '{csv_filename}'.")

Loading evaluation dataset...
Loading Whisper 'medium' model...

Starting Transcription and Evaluation...

[HIN_01] WER: 0.600 | CER: 0.267
[HIN_02] WER: 0.556 | CER: 0.156
[HIN_03] WER: 0.812 | CER: 0.257
[HIN_04] WER: 0.636 | CER: 0.224
[HIN_05] WER: 0.462 | CER: 0.161
[HIN_06] WER: 0.750 | CER: 0.345
[HIN_07] WER: 0.583 | CER: 0.167
[HIN_08] WER: 0.500 | CER: 0.213
[HIN_09] WER: 0.571 | CER: 0.273
[HIN_10] WER: 0.286 | CER: 0.057
[HIN_11] WER: 0.357 | CER: 0.141
[HIN_12] WER: 0.769 | CER: 0.299
[HIN_13] WER: 0.385 | CER: 0.113
[HIN_14] WER: 0.429 | CER: 0.156
[HIN_15] WER: 0.533 | CER: 0.223
[HIN_16] WER: 0.529 | CER: 0.139
[HIN_17] WER: 0.571 | CER: 0.250
[HIN_18] WER: 0.786 | CER: 0.282
[HIN_19] WER: 0.467 | CER: 0.208
[HIN_20] WER: 0.733 | CER: 0.254

🏆 MMS EVALUATION SUMMARY
Average WER: 0.566
Average CER: 0.209

✅ Detailed breakdown saved to 'mms_whisper_evaluation_report.csv'.


In [ ]:
import shutil
from google.colab import files

print("Zipping the audio files...")
# Compress the folder into a file named 'xtts_audio_results.zip'
shutil.make_archive('meta-mms_audio_results', 'zip', 'mms_hindi_eval')

print("Starting download...")
# Trigger the browser download
files.download('meta-mms_audio_results.zip')

Zipping the audio files...
Starting download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>